# 06 External Validation

Run in Google Colab only. Outputs must be saved in the matching results folder.


In [ ]:
from google.colab import drive
drive.mount("/users/")


In [ ]:
import os, sys
from pathlib import Path

def find_sparseguard_project_root():
    env_root = os.environ.get("SPARSEGUARD_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/users/"),
        Path("/users/"),
        Path.cwd(),
    ])
    drive_root = Path("/users/")
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob("IMPLEMENTATION/src/sparseguard_pipeline.py"))
    valid = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        if (candidate / "src" / "sparseguard_pipeline.py").exists():
            score = 0
            score += 10 if "LocalDrive1/OnlyScholar/Projects" in str(candidate) else 0
            score += 5 if (candidate / "EXPERIMENT" / "results" / "sparseguard_best.pt").exists() else 0
            score += 3 if (candidate / "Q1_VALIDATION" / "scripts" / "q1_sci_validation.py").exists() else 0
            valid.append((score, candidate))
    if not valid:
        raise FileNotFoundError("Could not locate finalized IMPLEMENTATION/src/sparseguard_pipeline.py. Mount Drive or set SPARSEGUARD_ROOT.")
    return sorted(valid, key=lambda item: (item[0], len(str(item[1]))), reverse=True)[0][1]

def find_dataset_root():
    env_root = os.environ.get("SPARSEGUARD_DATASET_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/users/"),
        Path("/users/"),
    ])
    drive_root = Path("/users/")
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob("X-IIoTID/X-IIoTID dataset.csv"))
    for candidate in candidates:
        if (candidate / "X-IIoTID" / "X-IIoTID dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate X-IIoTID dataset root. Mount Drive or set SPARSEGUARD_DATASET_ROOT.")

PROJECT_ROOT = find_sparseguard_project_root()
DATASET_ROOT = find_dataset_root()
os.environ["SPARSEGUARD_ROOT"] = str(PROJECT_ROOT)
os.environ["SPARSEGUARD_DATASET_ROOT"] = str(DATASET_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATASET_ROOT", DATASET_ROOT)


In [ ]:
%run {PROJECT_ROOT / "EVALUATION/scripts/run_external_validation.py"}


## After Run Documentation
Write a human summary in `DOCUMANTATION/` with inputs, outputs, metrics, curves, runtime, FLOPs, energy, warnings, and paper-use notes.
